SHOPPING DATASET CLEANING

Step 1 - Load Dataset


In [ ]:
import pandas as pd

df = pd.read_csv('Combined_dataset.csv')
print("Dataset loaded successfully!")

Step 2 - EDA(Exploratory Data Analysis)

In [ ]:
# First 5 rows — quick look at the data
df.head()

In [ ]:
# Last 5 rows — check if data is consistent at the end
df.tail()

In [ ]:
# Shape — how big is this dataset?
print("Shape:", df.shape)
print(f"{df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# Column names
print("Columns:", df.columns.tolist())

In [ ]:
# Data types of each column
# object = text/string, int64 = whole numbers, float64 = decimal numbers
print(df.dtypes)

In [ ]:
# df.info() —Shows column names, how many non-null values exist, and data types
# If Non-Null Count < total rows → that column has missing values
df.info()

In [ ]:
# Statistical summary — only for numeric columns
# Shows count, mean, std, min, max, and percentiles
df.describe()

In [ ]:
# Unique categories in the dataset
print("Categories:")
print(df['category'].value_counts())

In [ ]:
# Unique values per column — useful for finding messy columns
for col in df.columns:
    print(f"  {col:<30}: {df[col].nunique()} unique values")

Step 3 - Handling Missing values

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Check what the raw final_price values actually look like
print(df['final_price'].head(10).tolist())

In [ ]:
# final_price is empty in this dataset
# Calculate it from initial_price and discount
# Formula: final_price = initial_price - (initial_price * discount / 100)

df['final_price'] = df['initial_price'] - (df['initial_price'] * df['discount'] / 100)

print("'final_price' calculated from initial_price and discount")
print(df[['initial_price', 'discount', 'final_price']].head())

In [ ]:
# Fill missing numeric values with median
# Median is better than mean (not affected by extreme prices)

for col in ['initial_price', 'final_price', 'rating', 'discount']:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  '{col}' — filled {df[col].isnull().sum()} nulls with median ({median_val:.2f})")

In [ ]:
# Fill missing text columns with 'Unknown'
text_fill_cols = ['seller_name', 'category', 'sizes', 'currency']

for col in text_fill_cols:
    if df[col].isnull().sum() > 0:
        n = df[col].isnull().sum()
        df[col] = df[col].fillna('Unknown')
        print(f"  '{col}' — filled {n} nulls with 'Unknown'")

In [ ]:
# Confirm
print("Missing values after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal nulls remaining: {df.isnull().sum().sum()}")

Step 4 - Filter Rows & selet columns

In [ ]:
# Select useful columns only
useful_cols = ['product_id', 'title', 'category', 'rating',
               'ratings_count', 'initial_price', 'discount',
               'final_price', 'seller_name', 'sizes']

df_clean = df[useful_cols]
df_clean.head()

In [ ]:
# Filter — products with rating above 4.0
high_rated = df_clean[df_clean['rating'] > 4.0]
print(f"High rated products (>4.0): {len(high_rated)}")
high_rated.head()

In [ ]:
# Filter — products with discount above 20%
high_discount = df_clean[df_clean['discount'] > 20]
print(f"Products with >20% discount: {len(high_discount)}")
high_discount.head()

In [ ]:
# Combined filter — high rating AND high discount
good_deal = df_clean[(df_clean['rating'] > 4.0) & (df_clean['discount'] > 20)]
print(f"High rated + high discount: {len(good_deal)}")
good_deal.head()

Step 5 - Remove duplicates
# Check for duplicates based on 'product_id'

In [ ]:
print(f"Duplicate rows found: {df_clean.duplicated().sum()}")

In [ ]:
print(f"Rows before: {len(df_clean)}")

df_clean = df_clean.drop_duplicates(keep='first')
df_clean = df_clean.reset_index(drop=True)

print(f"Rows after:  {len(df_clean)}")
print("Duplicates removed ✅")

Step 6 - Derived Columns

In [ ]:
# savings = how much money the customer saves per product
# This replaces total_amount = price × quantity (no quantity column in this dataset)

df_clean['savings'] = df_clean['initial_price'] - df_clean['final_price']

print("'savings' column created ✅")
df_clean[['title', 'initial_price', 'final_price', 'discount', 'savings']].head(8)

In [ ]:
# Insights from the new column
print(f"Average savings per product : ₹{df_clean['savings'].mean():,.2f}")
print(f"Maximum savings             : ₹{df_clean['savings'].max():,.2f}")
print(f"Minimum savings             : ₹{df_clean['savings'].min():,.2f}")

In [ ]:
# Average savings by category
print("Average Savings by Category:")
print(df_clean.groupby('category')['savings']
      .mean()
      .sort_values(ascending=False)
      .round(2))

Final Step - Save the dataset

In [ ]:
df_clean.to_csv('cleaned_shopping_dataset.csv', index=False)

print("Saved as 'cleaned_shopping_dataset.csv' ✅")
print(f"Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")